# 🎵 ACE-Step UI — GPU (Google Colab · T4)

Запуск проекта **ace-step-ui** на бесплатном GPU **NVIDIA T4** с автосохранением готовых треков на ваш **Google Drive**.

## Порядок действий
1. Меню **Среда выполнения → Сменить среду выполнения → T4 GPU**.
2. Выполняйте ячейки сверху вниз (Shift+Enter).
3. Когда Colab попросит — разрешите доступ к Google Drive (треки сохраняются в папку `MyDrive/ACE-Step-Output`).
4. В конце откройте публичную ссылку `*.trycloudflare.com`.

### 🎯Выбор модели (качество)
В ячейке 7 переменная `DIT_MODEL`:
- `acestep-v15-turbo` — быстрая (8 шагов), по умолчанию.
- `acestep-v15-xl-turbo` — **макс. качество** на T4 (4B DiT, 8 шагов, ~9GB с offload).
- `acestep-v15-xl-sft` — абсолютный максимум (50 шагов + CFG, медленно).

In [ ]:
!nvidia-smi

In [ ]:
# 1) Node.js 20 + ffmpeg + cloudflared
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs build-essential ffmpeg > /dev/null 2>&1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cf.deb && sudo dpkg -i /tmp/cf.deb > /dev/null 2>&1
!node -v && cloudflared --version

In [ ]:
# 2) Клонируем UI и движок; ставим движок
# ВАЖНО: pyproject движка пинит torch с кастомного индекса (только для uv),
# от чего обычный 'pip install -e .' падает. Ставим пакет без deps
# (CUDA-torch в Colab уже есть), а остальные библиотеки ставим с ЗАКРЕПЛЁННЫМИ версиями
# (движок НЕСОВМЕСТИМ с transformers 5.x — нужен 4.x!).
%cd /content
![ -d ace-step-ui_CPU ] || git clone -q https://github.com/Landers125/ace-step-ui_CPU.git
![ -d ACE-Step-1.5 ] || git clone -q https://github.com/ace-step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!pip install -q -e . --no-deps
!pip install -q -e acestep/third_parts/nano-vllm --no-deps
!pip install -q "transformers>=4.51.0,<4.58.0" "diffusers>=0.37.0" "accelerate>=1.12.0" "soundfile>=0.13.1" loguru einops scipy "vector-quantize-pytorch>=1.27.15" diskcache numba pytorch-wavelets pywavelets toml modelscope matplotlib librosa soxr
import torch, transformers; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| transformers', transformers.__version__)

### Если выше показано `CUDA: False`
Выполните следующую ячейку (переустановит PyTorch с CUDA). Если `CUDA: True` — **пропустите** её.

In [ ]:
# (только при CUDA: False)
!pip install -q --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu121
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
# 3) Зависимости интерфейса (frontend + backend)
%cd /content/ace-step-ui_CPU
!npm install --silent
!cd server && npm install --silent
print('UI deps installed')

In [ ]:
# 4) Патчи: Vite allowedHosts, CORS бэкенда (dev), выбор DiT-модели
import pathlib, os
vc = pathlib.Path('/content/ace-step-ui_CPU/vite.config.ts')
s = vc.read_text()
anchor = 'port: 3000,'
if 'allowedHosts' not in s:
    s = s.replace(anchor, anchor + chr(10) + '      allowedHosts: true,')
    vc.write_text(s)
print('allowedHosts patched:', 'allowedHosts' in vc.read_text())

ix = pathlib.Path('/content/ace-step-ui_CPU/server/src/index.ts')
t = ix.read_text()
needle = "if (config.nodeEnv === 'development') {"
if 'dev-allow-all' not in t:
    t = t.replace(needle, needle + ' return callback(null, true); // dev-allow-all', 1)
    ix.write_text(t)
print('CORS dev-allow-all patched:', 'dev-allow-all' in ix.read_text())

# делаем DiT-модель управляемой через ACESTEP_CONFIG_PATH
sg = pathlib.Path('/content/ace-step-ui_CPU/server/scripts/simple_generate.py')
g = sg.read_text()
if 'ACESTEP_CONFIG_PATH' not in g:
    g = g.replace('config_path="acestep-v15-turbo",', 'config_path=os.environ.get("ACESTEP_CONFIG_PATH", "acestep-v15-turbo"),')
    sg.write_text(g)
print('DiT model switchable:', 'ACESTEP_CONFIG_PATH' in sg.read_text())

In [ ]:
# 5) Публичный туннель cloudflared на порт 3000
import subprocess, time, re
subprocess.Popen('cloudflared tunnel --url http://localhost:3000 --no-autoupdate > /content/cf.log 2>&1', shell=True)
url = None
for _ in range(40):
    time.sleep(2)
    try:
        log = open('/content/cf.log').read()
    except Exception:
        log = ''
    m = re.search('https://[a-z0-9-]+[.]trycloudflare[.]com', log)
    if m:
        url = m.group(0)
        break
print('PUBLIC URL:', url or 'not found yet — see /content/cf.log')

In [ ]:
# 6) Подключаем Google Drive (готовые треки будут автосохраняться сюда)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/ACE-Step-Output'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Готовые треки будут сохраняться в:', DRIVE_DIR)

In [ ]:
# 7) Пишем .env и запускаем backend + frontend
import sys, os, subprocess, time
root = '/content/ace-step-ui_CPU'
public = url if ('url' in dir() and url) else 'http://localhost:3000'
# DiT-модель. Для макс. качества на T4 поставьте 'acestep-v15-xl-turbo'
DIT_MODEL = 'acestep-v15-turbo'
env_text = ('NODE_ENV=development' + chr(10) + 'PORT=3001' + chr(10) + 'FRONTEND_URL=' + public + chr(10) + 'ACESTEP_API_URL=http://localhost:8001' + chr(10) + 'ACESTEP_PATH=/content/ACE-Step-1.5' + chr(10) + 'ACESTEP_CONFIG_PATH=' + DIT_MODEL + chr(10) + 'PYTHON_PATH=' + sys.executable + chr(10))
open(root + '/.env', 'w').write(env_text)
e = os.environ.copy()
e['ACESTEP_CONFIG_PATH'] = DIT_MODEL
subprocess.Popen('npx tsx src/index.ts > /content/backend.log 2>&1', shell=True, cwd=root + '/server', env=e)
subprocess.Popen('npm run dev > /content/frontend.log 2>&1', shell=True, cwd=root, env=e)
print('Запуск сервисов (модель: ' + DIT_MODEL + '), ждём 30 секунд...')
time.sleep(30)
print(open('/content/backend.log').read()[-1500:])
print('----- FRONTEND -----')
print(open('/content/frontend.log').read()[-800:])
print('OPEN:', public)

In [ ]:
# 8) Автосохранение готовых треков в Google Drive по подпапкам с датой (фоновый процесс)
import threading, time, shutil, os, glob, datetime
AUDIO_DIR = '/content/ace-step-ui_CPU/server/public/audio'
DRIVE_DIR = '/content/drive/MyDrive/ACE-Step-Output'
os.makedirs(DRIVE_DIR, exist_ok=True)

def _sync_loop():
    while True:
        try:
            for f in glob.glob(os.path.join(AUDIO_DIR, '*')):
                if not os.path.isfile(f):
                    continue
                if f.lower().endswith(('.mp3', '.flac', '.wav')):
                    name = os.path.basename(f)
                    day = datetime.date.fromtimestamp(os.path.getmtime(f)).isoformat()
                    day_dir = os.path.join(DRIVE_DIR, day)
                    os.makedirs(day_dir, exist_ok=True)
                    dest = os.path.join(day_dir, name)
                    if not os.path.exists(dest):
                        try:
                            shutil.copy2(f, dest)
                            print('Saved to Drive:', day + '/' + name)
                        except Exception as ex:
                            print('Copy error', name, ex)
        except Exception:
            pass
        time.sleep(15)

threading.Thread(target=_sync_loop, daemon=True).start()
print('Автосохранение в Google Drive (по датам) запущено. Папка: ' + DRIVE_DIR)

## ✅ Готово
- Откройте ссылку `https://....trycloudflare.com` из ячейки выше.
- **Готовые треки автоматически копируются** в `Google Drive → MyDrive → ACE-Step-Output → ГГГГ-ММ-ДД`.
- **Первая генерация** скачивает веса модели — разовая задержка (XL — ~9GB).
- На T4 держите **Batch Size = 1**, длительность 30–120 сек.
- Логи генерации: `!tail -n 60 /content/backend.log`.